# Retention Analysis — Tidemill vs Stripe

Cohort retention analysis comparing the Tidemill engine with Stripe
subscription lifecycle data.

## How cohort retention is computed

1. **Cohort assignment:** each customer is assigned to the month of their
   *earliest* `subscription.created` timestamp.

2. **Monthly activity:** a customer is "active" in month M if they have at
   least one subscription that was created on or before M and whose
   `canceled_at` is either null or falls after M.

3. **Retention rate** for each (cohort, month-offset) cell:

   ```
   retention = active_customers_in_cell / cohort_size × 100
   ```

## Revenue retention (NRR / GRR)

- **NRR** (Net Revenue Retention) = (start MRR + expansion - contraction - churn) / start MRR
  - NRR > 100% means expansion outpaces churn
- **GRR** (Gross Revenue Retention) = (start MRR - contraction - churn) / start MRR
  - GRR is always <= 100% (ignores expansion)

## Stripe fields used

- `subscription.created` — determines cohort assignment
- `subscription.canceled_at` — determines when a customer leaves a month
- `subscription.customer` — groups subscriptions into customer entities

In [1]:
import os

import stripe

from tidemill import reports
from tidemill.reports.stripecheck import StripeData, TidemillClient
from tidemill.reports.stripecheck import compare as cmp

stripe.api_key = os.environ["STRIPE_API_KEY"]
reports.setup()

START, END = "2025-09-01", "2026-03-31"
tm = TidemillClient()
sd = StripeData()

## 1. Cohort retention from Tidemill API

In [2]:
ret_matrix = reports.retention.stripe_heatmap(sd, START, END)
reports.retention.plot_stripe_heatmap(ret_matrix)

## 2. Cohort retention from Stripe (ground truth)

`stripecheck.stripe_metrics.cohort_retention()` builds the full retention
matrix directly from Stripe subscription objects:

1. Groups subs by `customer`, finds earliest `created_at` -> cohort month
2. For each (customer, calendar month), checks if any sub is still active
3. Pivots into a cohort x month-offset matrix of retention percentages

In [3]:
cohort_sizes = ret_matrix.attrs["cohort_sizes"]
print("Cohort sizes (customers per signup month):")
print(cohort_sizes.to_string())
print(f"\nRetention matrix: {ret_matrix.shape[0]} cohorts × {ret_matrix.shape[1]} months")

Cohort sizes (customers per signup month):
cohort_month
2025-09    20
2025-10     5
2025-11     3
2025-12     5
2026-01     5

Retention matrix: 5 cohorts × 7 months


## 3. Cohort retention heatmap (Stripe)

In [4]:
reports.retention.style_stripe_retention(ret_matrix)

months_since,0,1,2,3,4,5,6
cohort_month,,,,,,,
2025-09,90%,70%,70%,70%,70%,70%,70%
2025-10,100%,80%,80%,80%,80%,80%,0%
2025-11,100%,33%,33%,33%,33%,0%,0%
2025-12,100%,80%,80%,80%,0%,0%,0%
2026-01,100%,60%,60%,0%,0%,0%,0%


## 4. Compare Tidemill retention with Stripe

`stripecheck.compare.retention()` merges both sources into a single
DataFrame so you can inspect per-cell divergences.

In [5]:
cmp.retention(tm, sd, START, END)

,cohort_month,active_month,cohort_size,active_count,retention_pct,stripe_active,stripe_cohort_size,stripe_retention_pct
0,2025-09,2025-09,0.0,0.0,0.0,18.0,20.0,90.000000
1,2025-09,2025-10,0.0,0.0,0.0,14.0,20.0,70.000000
2,2025-09,2025-11,0.0,0.0,0.0,14.0,20.0,70.000000
3,2025-09,2025-12,0.0,0.0,0.0,14.0,20.0,70.000000
4,2025-09,2026-01,0.0,0.0,0.0,14.0,20.0,70.000000
5,2025-09,2026-02,0.0,0.0,0.0,14.0,20.0,70.000000
6,2025-09,2026-03,0.0,0.0,0.0,14.0,20.0,70.000000
7,2025-09-01,2025-09-01,20.0,20.0,100.0,0.0,0.0,0.000000
8,2025-09-01,2025-10-01,6.0,6.0,100.0,0.0,0.0,0.000000
9,2025-10,2025-10,0.0,0.0,0.0,5.0,5.0,100.000000


## 5. Retention curve

In [6]:
reports.retention.plot_stripe_curve(ret_matrix.mean(axis=0))

## 6. Monthly NRR and GRR

Net Revenue Retention and Gross Revenue Retention per month from the Tidemill API.

- **NRR** = (start MRR + expansion - contraction - churn) / start MRR
- **GRR** = (start MRR - contraction - churn) / start MRR

NRR > 100% means expansion outpaces churn. GRR is always <= 100%.

In [7]:
rr = reports.retention.nrr_grr(tm, START, END)
reports.retention.style_nrr_grr(rr)

,nrr,grr
month,,
2025-09,nan%,nan%
2025-10,103.2%,79.8%
2025-11,85.6%,85.6%
2025-12,95.0%,95.0%
2026-01,97.8%,97.8%
2026-02,95.8%,95.8%


In [8]:
reports.retention.plot_nrr_grr(rr)